# CORE-ave: Model Definition

Implementation of CORE-ave (mean pooling variant) as described in:
*CORE: Simple and Effective Session-based Recommendation within Consistent Representation Space* (SIGIR 2022)

Key components:
1. Item embedding table (shared for encoding and decoding)
2. Masked mean pooling over valid positions
3. L2-normalized session and item vectors
4. Cosine similarity with temperature scaling
5. InfoNCE loss with controllable margin

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
import math

In [ ]:
class CoreAve(nn.Module):
    """
    CORE-ave: Representation-Consistent Session Encoder with Mean Pooling.

    Session embedding = mean of clicked item embeddings (masked for padding)
    Score = cosine_sim(session_emb, item_emb) / temperature
    Loss = InfoNCE with controllable margin
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 100,
        padding_idx: int = 0,
        item_dropout: float = 0.1,
        init_temperature: float = 1.0,
        margin: float = 0.07,
        min_temperature: float = 0.01
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.padding_idx = padding_idx
        self.margin = margin
        self.min_temperature = min_temperature

        # Item embedding table (shared for encoding and decoding)
        self.item_embeddings = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=padding_idx
        )

        # Item dropout for robust distance measuring (paper)
        self.item_dropout = nn.Dropout(item_dropout)

        # Learnable temperature parameter
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temperature)))

        self._init_weights()

    def _init_weights(self):
        """Initialize embeddings with normal distribution."""
        nn.init.normal_(self.item_embeddings.weight, mean=0.0, std=0.01)
        with torch.no_grad():
            self.item_embeddings.weight[self.padding_idx].fill_(0)

    @property
    def temperature(self) -> torch.Tensor:
        """Temperature with minimum bound."""
        return torch.clamp(self.log_temperature.exp(), min=self.min_temperature)

    def encode_session(self, prefix_indices: torch.Tensor, prefix_lengths: torch.Tensor) -> torch.Tensor:
        """
        Encode session as masked mean of item embeddings.

        Args:
            prefix_indices: [batch_size, max_len] - item indices (0 = padding)
            prefix_lengths: [batch_size] - actual lengths

        Returns:
            session_emb: [batch_size, embedding_dim] - L2 normalized
        """
        batch_size, max_len = prefix_indices.shape

        # Get item embeddings: [batch_size, max_len, embedding_dim]
        item_embs = self.item_embeddings(prefix_indices)

        # Apply item dropout during training
        if self.training:
            item_embs = self.item_dropout(item_embs)

        # Create mask for valid positions: [batch_size, max_len]
        mask = (prefix_indices != self.padding_idx).float()

        # Masked mean pooling
        masked_sum = (item_embs * mask.unsqueeze(-1)).sum(dim=1)
        lengths = prefix_lengths.float().unsqueeze(-1).clamp(min=1.0)
        session_emb = masked_sum / lengths

        # L2 normalize for cosine similarity
        session_emb = F.normalize(session_emb, p=2, dim=-1)

        return session_emb

    def forward(
        self,
        prefix_indices: torch.Tensor,
        prefix_lengths: torch.Tensor,
        target_indices: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass.

        Args:
            prefix_indices: [batch_size, max_len]
            prefix_lengths: [batch_size]
            target_indices: [batch_size] - for training loss computation

        Returns:
            scores: [batch_size, vocab_size] - cosine similarity / temperature
            loss: scalar (if target_indices provided) or None
        """
        # Encode session
        session_emb = self.encode_session(prefix_indices, prefix_lengths)

        # Get all item embeddings for scoring
        all_item_embs = self.item_embeddings.weight
        all_item_embs_norm = F.normalize(all_item_embs, p=2, dim=-1)

        # Cosine similarity with temperature
        scores = torch.matmul(session_emb, all_item_embs_norm.t()) / self.temperature

        # Mask padding item (index 0)
        mask_value = -1e4 if scores.dtype == torch.float16 else -1e9
        scores[:, self.padding_idx] = mask_value

        loss = None
        if target_indices is not None:
            loss = self.compute_loss(scores, target_indices)

        return scores, loss

    def compute_loss(self, scores: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        InfoNCE loss with controllable margin.

        L = -log(exp(s_pos + m) / (exp(s_pos + m) + sum(exp(s_neg))))
        where m is the margin.
        """
        batch_size = scores.shape[0]

        pos_scores = scores[torch.arange(batch_size), targets]
        pos_scores_margin = pos_scores + self.margin
        log_sum_exp = torch.logsumexp(scores, dim=1)
        loss = (log_sum_exp - pos_scores_margin).mean()

        return loss

    def recommend(
        self,
        prefix_indices: torch.Tensor,
        prefix_lengths: torch.Tensor,
        k: int = 20
    ) -> torch.Tensor:
        """Get top-k recommendations for a batch of prefixes."""
        self.eval()
        with torch.no_grad():
            scores, _ = self.forward(prefix_indices, prefix_lengths, target_indices=None)
            top_k = torch.topk(scores, k=k, dim=1)
            return top_k.indices

    def get_item_embeddings(self) -> torch.Tensor:
        """Get normalized item embeddings for external use."""
        with torch.no_grad():
            embs = self.item_embeddings.weight.clone()
            embs = F.normalize(embs, p=2, dim=-1)
            return embs


def create_core_ave(vocab_size: int, **kwargs) -> CoreAve:
    """Factory function for CORE-ave."""
    return CoreAve(vocab_size=vocab_size, **kwargs)

In [ ]:
# Model Summary & Parameter Count
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

VOCAB_SIZE = 34644  # From data preparation
model = create_core_ave(vocab_size=VOCAB_SIZE).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Embedding table: {model.item_embeddings.weight.shape}")

# Parameter breakdown
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape} ({param.numel():,} params)")

In [ ]:
# Forward Pass Test
batch_size = 4
max_len = 50

# Dummy batch
prefix = torch.randint(1, 100, (batch_size, max_len)).to(device)
prefix[:, :20] = 0  # Left padding
lengths = torch.randint(5, 30, (batch_size,)).to(device)
targets = torch.randint(1, 1000, (batch_size,)).to(device)

model.train()
scores, loss = model(prefix, lengths, targets)

print(f"Scores shape: {scores.shape}")  # [4, 34644]
print(f"Loss: {loss.item():.4f}")
print(f"Temperature: {model.temperature.item():.4f}")

# Check padding mask
print(f"Score at padding idx: {scores[0, 0].item()}")
print(f"Score min: {scores.min().item():.4f}")
print(f"Score max: {scores.max().item():.4f}")

In [ ]:
# Recommendation Test
model.eval()
with torch.no_grad():
    topk = model.recommend(prefix, lengths, k=20)
    print(f"Top-20 shape: {topk.shape}")
    print(f"Sample recommendations (first batch): {topk[0].tolist()}")

# Verify padding not in recommendations
for i in range(batch_size):
    assert 0 not in topk[i].tolist(), f"Padding found in recommendations for batch {i}"
print("No padding in recommendations: OK")

In [ ]:
# Gradient Flow Test
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for step in range(3):
    optimizer.zero_grad()
    scores, loss = model(prefix, lengths, targets)
    loss.backward()

    # Check gradients
    emb_grad = model.item_embeddings.weight.grad
    temp_grad = model.log_temperature.grad

    print(f"Step {step+1}: Loss={loss.item():.4f}")
    print(f"  Emb grad norm: {emb_grad.norm().item():.4f}")
    print(f"  Temp grad: {temp_grad.item():.6f}")

    optimizer.step()

print("Gradient flow test passed")

In [ ]:
# Save Model Checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': 100,
    'padding_idx': 0,
    'margin': 0.07,
}

torch.save(checkpoint, "core_ave_model.pt")
print("Model saved to core_ave_model.pt")

# Verify loading
model2 = create_core_ave(vocab_size=VOCAB_SIZE)
model2.load_state_dict(torch.load("core_ave_model.pt")['model_state_dict'])
model2.to(device)
model2.eval()

with torch.no_grad():
    scores2, _ = model2(prefix, lengths)
    print(f"Scores match: {torch.allclose(scores, scores2)}")